# Modelos Escondidos de Markov e as suas aplicações na Bioinformática

## Intro

Um modelo de Markov é um modelo estocástico que visa representar sistemas que mudam de estado de forma pseudo-aleatória. Neste notebook, damos-te algumas ferramentas para compreender o funcionamento deste modelo estatístico e um exemplo real da sua aplicação

---

###### dependências: python 3.13 - pela sintaxe de typing utilizada, numpy, scipy, pandas, biopython, (seaborn caso queiramos visualizar as matrizes como heatmaps)

In [28]:
from __future__ import annotations

import sys
import numpy as np
import pandas as pd
import seaborn as sns

import Bio.SeqIO as SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils import IUPACData

from my_types import *
from aux_functions import *
from markov_clases import *

### Cadeia de Markov

O modelo de Markov mais simples é a **Cadeia de Markov**.

Este modelo, assim como todos os outros Modelos de Markov, assumem a **Propriedade de Markov**, que diz que o que acontece no futuro depende unicamente da situação no presente, independentemente do que aconteceu no passado. Formalmente:
<big> 
$$ 
P(X_{t+1}|X_{t}, X_{t-1}, X_{t-2}, ...) = P(X_{t+1}|X_{t})
$$ 
</big>

Uma Cadeia de Markov é frequentemente representada como um grafo direcionado, onde os nós representam os estados possíveis da sequência modelada e as arestas têm valores associados que representam a probabilidade de que o estado alvo se segue ao estado de origem. Estas probabilidades designam-se **probabilidades de transição**.

<big> Seja:  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$S$ o espaço de estados possíveis e  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$n=\#S$,  

as probabilidades de transição podem ser agrupadas numa matriz  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$A_{n \times n}$  

designada **matriz de transição**, onde   
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$A_{ij} = P(i \rightarrow j) $  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;ou seja, a probabilidade do estado $j$ aparecer a seguir ao estado $i$ 
</big>

Tendo em conta a *Lei dos Grandes Números*, se percorrermos a cadeia aleatoriamente e registarmos as vezes que passámos por cada estado, eventualmente, as proporções estabilizam. Ao vetor dessas proporções estáveis chamamos **Distribuição Estacionária** da Cadeia e é usualmente denotada por $\pi$. Para a calcular, apresentamos 3 métodos:
- O método descrito designa-se por método Monte Carlo

- O 2º método baseia-se na propriedade de que <br><big>
$P(X_1=i, X_t=j) = A^t_{ij}$ </big><br>logo, $\pi$ é igual a qualquer linha de $A^{+\infty}$ 

- O 3º método baseia-se no facto de que queremos que os valores de $\pi$ sejam constantes independentemente dos estados que percorremos, ou seja, queremos que <br><big>
$\pi \cdot A = \pi$ </big><br> Se se lembram das aulas de ALGA, esta igualdade diz-nos que $\pi$ é um *vetor próprio* (esquerdo) de $A$ associado ao valor próprio $1$. depois de o calcular, normalizamo-lo para que a sua soma seja igual a 1, afinal, estamos a trabalhar com probabilidades 

###### *De facto, exatamente por estarmos a trabalhar com probabilidades, todas as linhas das matrizes que definirmos têm que ter uma soma de 1*

Frequentemente, as condições iniciais de uma Cadeia de Markov são específicas e diferentes do estado estacionário, mas também representadas por um vetor $\pi$, então é importante distinguir se $\pi$ se refere à **Distribuição Estacionária** ou à **Distribuição *Inicial***

---

### classe MarkovChain

Aqui podes explorar a classe `MarkovChain`  
inicializando uma cadeia com uma matriz de transição, e, opcionalmente, uma lista de estados e uma distribuição inicial  

A partir daí podes experimentar fazer random walks e calcular a distribuição estacionária da cadeia com os diferentes métodos

In [29]:
seed = 78
# exemplo: fazer a previsão do tempo para o próximo mês
mc = MarkovChain([[0.2, 0.6, 0.2], 
                  [0.3, 0.0, 0.7], 
                  [0.5, 0.0, 0.5]],
                 ["nuvens","chuva","sol"])
a=mc.random_walk(30, seed=seed)

nuvens → chuva → sol → sol → sol → nuvens → sol → nuvens → nuvens →

chuva → sol → sol → sol → nuvens → chuva → sol → nuvens → sol → sol →

sol → nuvens → sol → nuvens → nuvens → chuva → sol → sol → nuvens →

chuva → sol


In [30]:
print(mc.indexed_matrix)
mul =mc.calculate_pi("RepMatMul")
car =mc.calculate_pi("MonteCarlo", seed=seed)
eig =mc.calculate_pi("LeftEigVec")
print()
print("car    ", car)
print("mul    ", mul)
print("eig    ", eig)
print()
print("mul/car", mul/car)
print("car/eig", car/eig)
print("eig/mul", eig/mul)

        nuvens  chuva  sol
nuvens     0.2    0.6  0.2
chuva      0.3    0.0  0.7
sol        0.5    0.0  0.5

car     [0.3535 0.2094 0.4371]
mul     [0.35211268 0.21126761 0.43661972]
eig     [0.35211268 0.21126761 0.43661972]

mul/car [0.99607546 1.00891884 0.99890121]
car/eig [1.00394 0.99116 1.0011 ]
eig/mul [1. 1. 1.]


---
---

### Hidden Markov Model

Um **Modelo Escondido/Oculto de Markov (HMM)** tem como base uma Cadeia de Markov, a diferença é que, para além das probabilidades de transição entre estados, cada estado tem outro conjunto de probabilidades, as **probabilidades de emissão** de *símbolos* associados ao HMM. Nos sistemas que são representados por HMMs, esses símbolos são a única forma de nós, observadores, medirmos o sistema. Por isso, os estados da Cadeia associada são chamados de **Estados Escondidos/Ocultos**(**Hidden States**), daí o nome ***Hidden* Markov Model**

<big> Seja:   
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$V$ o espaço dos símbolos observáveis,  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$m=\#V$ e  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$v$ um símbolo tal que $v \in V$   

as probabilidades de emissão também podem ser organizadas numa **matriz de Emissão**  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$B_{m\times n}$  

onde  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$B_{iv} = P(v|i)$  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;ou seja, a probabilidade do símbolo $v$ ser uma emissão do estado $i$
</big>

A utilidade destes modelos reside na simplicidade originada pela Propriedade de Markov de desenvolver algoritmos que nos permitem descobrir as probabilidades de ocorrência das sequências tanto de símbolos observáveis como de estados ocultos. Todos os que vamos explorar são algoritmos de programação dinâmica, assumimos estes parâmetros:  

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$N =$ nº de estados  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$M =$ nº de símbolos  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$T =$ tamanho da sequência  

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$S = \{s_1, s_2, \dots, s_N\}$
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$X = \{x_1, x_2, \dots, x_T\}$  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$V = \{v_1, v_2, \dots, v_M\}$
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$Y = \{y_1, y_2, \dots, y_T\}$  

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$A$ - matriz de transição
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$B$ - matriz de emissão
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\pi$ - distribuição *inicial*  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\lambda = \{A, B, \pi\}$ - conjunto dos parâmetros do HMM

Para determinar a probabilidade de ocorrência de uma determinada sequência de símbolos, $P(Y|\lambda)$, temos dois algoritmos que podemos utilizar:

#### Forward Algorithm
Neste algoritmo temos uma matriz $\alpha$ onde $\alpha_{t}(i) = P(y_1, y_2, \dots, y_t\ \cap\ x_t = s_i\ \ |\ \ \lambda)$, isto é,  
a probabilidade de observar a sequência parcial $\{y_1,\dots,y_t\}$ e o estado oculto $s_i$ no tempo $t$, $t\in \{1,\dots,T\}$
<br>
<br>

$$
\begin{split}
        \alpha_{1}(i) = \pi_i \cdot B_{i, v_1}
        &
        \ \ \ \ \ \ \ \ ,\ i\in\{1,\dots,N\}
    \\
        \alpha_{t+1}(j) = \sum_{i=1}^N \alpha_t(i) \cdot A_{ij} \cdot B_{j, v_{t+1}}
        &
        \ \ \ \ \ \ \ \ ,\ j\in\{1,\dots,N\},\ t \in \{1,\dots, T\}
\end{split}
$$

$$P(Y|\lambda) = \sum_{i=1}^N \alpha_T(i)$$

#### Backward Algorithm
Este algoritmo é bastante semelhante ao forward mas começa pelo fim da sequência.  
Similarmente, temos uma matriz $\beta$ onde $\beta_{t}(i) = P(y_t, \dots, y_{T-1}, y_T\ \cap\ x_t = s_i\ \ |\ \ \lambda)$, isto é,  
a probabilidade de observar a sequência parcial $\{y_t,\dots,y_T\}$ e o estado oculto $s_i$ no tempo $t$, $t\in \{1,\dots,T\}$
<br>
<br>

$$
\begin{split}
        \beta_{T}(i) = 1 
        & 
        \ \ \ \ \ \ \ \ ,\ i\in\{1,\dots,N\}
    \\
        \beta_{t}(j) = \sum_{i=1}^N A_{ij} \cdot B_{j, v_{t+1}} \cdot \beta_{t+1}(i)
        &
        \ \ \ \ \ \ \ \ ,\ j\in\{1,\dots,N\},\ t \in \{1,\dots, T\}
\end{split}
$$

$$P(Y|\lambda) = \sum_{i=1}^N \pi_i \cdot B_{i, v_1} \cdot \beta_1(i)$$

Os dois algoritmos juntam-se no

#### Forward-Backward Algorithm
Com estas duas matrizes calculadas, podemos também calcular $P(x_t = s_i\ |\ Y, \lambda),\ i\in\{1,\dots,N\},\ t \in \{1,\dots, T\}$ e criar a matriz $\gamma$ onde $\gamma_t(i)$ representa então, a probabilidade de nos encontrarmos no estado $s_i$ no tempo $t$ tendo em conta a sequência observada e os parâmetros. Calculamo-la a partir do teorema de Bayes:
$$\gamma_t(i) = P(x_t = s_i\ |\ Y, \lambda) = \frac{P(x_t = s_i, Y\ |\ \lambda)}{P(Y\ |\ \lambda)}$$
$$\gamma_t(i) = \frac{\alpha_{t}(i) \cdot \beta_{t}(i)}{\sum_{j=1}^N \alpha_{t}(j) \cdot \beta_{t}(j)}\ \ \ \ \ \ \ \ ,\ i\in\{1,\dots,N\},\ t \in \{1,\dots, T\}$$

<br>

---

Estes algoritmos apenas nos dizem a probabilidade de uma sequência de símbolos ser observada, mas o maior propósito da utilização de HMMs é poder descobrir qual a sequência de estados ocultos que originou a sequência observada. Para isso, existe o 

#### Algoritmo de Viterbi

Este algoritmo utiliza-se da mesma técnica que o forward algorithm mas em vez da probabilidade total da sequência, na matriz $\delta$ (também denotada por $V$ em alguns lugares, mas aqui já usamos $V$ para o conjunto de símbolos), $\delta_t(i)$ representa a probabilidade *máxima*, considerando *todas* as sequências possíveis, do estado oculto em $t$ ser $s_i$. Adicionalmente, fazemos uma outra matriz $\psi$, onde $s_j,\ j = \psi_t(i)$ é o estado que maximiza a probabilidade $\delta_{t+1}(i)$, permitindo-nos reconstruir a sequência mais provável.
<br>
<br>

$$
\begin{split}
        \begin{split}
            \delta_{1}(i) &= \pi_i \cdot B_{i, v_1} 
            \\
            \psi_{1}(i) &= 0
        \end{split}
        &
        \ \ \ \ \ \ \ \ ,\ i\in\{1,\dots,N\}
    \\
    \\
        \begin{split}
            \delta_{t+1}(j) = \max_{1\le i \le N}\delta_t(i) \cdot A_{ij} \cdot B_{j, v_{t+1}} 
            \\
            \psi_{t+1}(j) = \argmax_{1\le i \le N}\delta_t(i) \cdot A_{ij} \cdot B_{j, v_{t+1}}
        \end{split}
        &
        \ \ \ \ \ \ \ \ ,\ j\in\{1,\dots,N\},\ t \in \{1,\dots, T\}
\end{split}
$$
$$
P^* = \max_{1\le i \le N}\delta_T(i)
\\
x_T^* =  \argmax_{1\le i \le N}\delta_T(i)
$$
<br>

$$
\begin{split}
    x_t^* = \psi_{t+1}(z)\ :\ x_{t+1}^* = s_z
    &
    \ \ \ \ \ \ \ \ ,\ t \in \{T-1,\dots, 1\}
\end{split}
$$
<br>

Terminamos, assim, com a sequência $X^* = \{x_1^*,\dots, x_T^*\}$ que é a sequência de estados ocultos com mais probabilidade de ter originado a sequência $O$

---

Caso não tenhamos qualquer ideia de como o sistema de estados ocultos funciona, também é possível treinar um HMM através de uma sequência ou conjunto de sequências observadas $Q$ 

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$R = \#Q =$ nº de sequências  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$Q = \{Y_1, \dots, Y_R\}$  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$Y_r = \{y_{r, 1}, \dots, y_{r, T_r}\},\ \ r \in \{1, \dots, R\} $  

para obter os parâmetros $\lambda = \{A, B, \pi\}$ que se melhor se adequam a $Q$ com o 

#### Algoritmo de Baum-Welch

Para este algoritmo temos que começar por definir um conjunto $\lambda$ aleatório ou definido com recurso a alguma informação prévia para "guiar" o algoritmo para o caminho certo. Com estes parâmetros iniciais, calculamos, para *todas* as sequências de $Q$ a matriz $\gamma$ com o Forward-Backward Algorithm:  

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\gamma_{t}(i) = P(x_t = s_i\ |\ Y, \lambda) = \frac{P(x_t = s_i,\ Y\ |\ \lambda)}{P(Y\ |\ \lambda)}$  

$$\gamma_{rt}(i) = \frac{\alpha_{rt}(i) \cdot \beta_{rt}(i)}{\sum_{j=1}^N \alpha_{rt}(j) \cdot \beta_{rt}(j)}\ \ \ \ \ \ \ \ ,\ i\in\{1,\dots,N\},\ t \in \{1,\dots, T_r\},\ r \in \{1, \dots, R\}$$


e uma nova matriz, $\xi$ onde $\xi_t(ij)$ representa a probabilidade de estar no estado $s_i$ no tempo $t$ *e* no estado $j$ no tempo $t+1$, tendo em conta a sequência observada e os parâmetros:  

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$\xi_t(ij) = P(x_t = s_i\ \cap \ x_{t+1} = s_j\ |\ Y, \lambda) = \frac{P(x_t = s_i\ \cap \ x_{t+1} = s_j,\ Y\ |\ \lambda)}{P(Y\ |\ \lambda)}$
<br>
<br>

$$\xi_{rt}(ij) = \frac{\alpha_{rt}(i) \cdot A_{ij} \cdot \beta_{r, t+1}(j) \cdot B_{j, v_{t+1}}}{\sum_{k=1}^N \sum_{w=1}^N \alpha_{rt}(k) \cdot A_{kw} \cdot \beta_{r,t+1}(w) \cdot B_{w, v_{t+1}}}\ \ \ \ \ \ \ \ ,\ i\in\{1,\dots,N\},\ t \in \{1,\dots, T_r\},\ r \in \{1, \dots, R\}$$

Com estas matrizes podemos atualizar os parâmetros do HMM:

$$~
\begin{split}
        \pi_i^* = \frac{\sum_{r=1}^R \gamma_{r,1}(i)}{R}&\ \ \ \ \ \ \ \ ,\ i\in \{1, \dots ,N\} 
    \\
    \\
        A_{ij}^* = \frac{\sum_{r=1}^R\sum_{t=1}^{T-1} \xi_{rt}(ij)}{\sum_{r=1}^R\sum_{t=1}^{T-1} \gamma_{rt}(i)}&\ \ \ \ \ \ \ \ ,\ i\in \{1, \dots ,N\},\ j\in \{1, \dots ,N\}
    \\
    \\
        B_{i,v_k}^* = \frac{\sum_{r=1}^R\sum_{t=1}^{T} \gamma_{rt}(i) \cdot 1_{y_{r,t} = v_k}}{\sum_{r=1}^R\sum_{t=1}^{T} \gamma_{rt}(i)}&\ \ \ \ \ \ \ \ ,\ i\in \{1, \dots ,N\},\ k\in \{1, \dots ,M\}
\end{split}
$$  

onde

$$1_{y_{r,t} = v_k} = 
\left\{
\begin{array}{ll}
  1 & se\ \ y_{r,t} = v_k
  \\
  0 & caso\ contrário
\end{array}
\right.
$$

Este processo então repete-se até os parâmetros convergirem

---

Ao multiplicar probabilidades (valores sempre entre 0 e 1) é normal que os valores cheguem a um ponto em que são tão pequenos que o computador desista, os valores dão underflow e todas as probabilidades passam a ser `0`. Um truque para contornar esse problema é fazer os cálculos em *espaço logarítmico*, onde essas probabilidades minúsculas passam a ser representadas por números negativos, e o domínio de valores que o computador consegue representar é extremamente maior.

---

### classe HiddenMarkovModel

Aqui podes explorar a classe `HiddenMarkovModel`  
inicializando um modelo com uma matriz de transição, e, opcionalmente, uma lista de estados e uma distribuição inicial (ou até uma `MarkovChain` que ja tenhas criado) juntamente com uma matriz de emissão e, opcionalmente, uma lista dos símbolos

A partir daí podes experimentar fazer random walks e aplicar os algoritmos acima

In [31]:
# exemplo: planear as saídas de casa do mês
hmm = HiddenMarkovModel(mc, [[0.2, 0.8],
                             [0.5, 0.5],
                             [1.0, 0.0]], values=("sair","ficar"))
hmm.random_walk(30, seed=seed)

 1  nuvens → chuva → sol  → sol  → sol  → nuvens → sol  → nuvens → 
 1  ↳sair    ↳ficar  ↳sair  ↳sair  ↳sair  ↳ficar   ↳sair  ↳ficar   

 8  nuvens → chuva → sol  → sol  → sol  → nuvens → chuva → sol  → 
 8  ↳sair    ↳sair   ↳sair  ↳sair  ↳sair  ↳ficar   ↳ficar  ↳sair  

16  nuvens → sol  → sol  → sol  → nuvens → sol  → nuvens → nuvens → 
16  ↳ficar   ↳sair  ↳sair  ↳sair  ↳ficar   ↳sair  ↳ficar   ↳sair    

24  chuva → sol  → sol  → nuvens → chuva → sol 
24  ↳sair   ↳sair  ↳sair  ↳sair    ↳sair   ↳sair  


In [32]:
# qual a probabilidade de ficar em casa uma semana inteira?
print(round(hmm.forward_algorithm(["ficar"]*7, log=False), 6))
print(round(hmm.backward_algorithm(["ficar"]*7, log=False), 6))

0.000902
0.000902


In [33]:
# Se na semana passada, saí de segunda a quinta, e fiquei em casa o resto dos dias, 
# qual foi, mais provavelmente, a meteorologia da semana?
saidas = ['sair']*4 + ["ficar"]*3
hmm.viterbi_algorithm(saidas)

Logaritmo da probablilidade: -2.803594154117529

1  sol  → sol  → sol  → sol  → nuvens → chuva → nuvens
1  ↳sair  ↳sair  ↳sair  ↳sair  ↳ficar   ↳ficar  ↳ficar   


---
---

## Contexto do problema 

Uma das aplicações dos HMMs em bioinformática é na previsão da topologia membranar de proteínas. Uma vez que, para determinar a sua estrutura, as proteínas são cristalizadas, o seu contexto celular perde-se. Ora, se não sabemos a sua posição relativamente à membrana podemos fazer um HMM que a descubra, tendo em conta informações que temos sobre a proteína, como, por exemplo, a sua sequência de aminoácidos! Neste caso os a.a. seriam os símbolos observáveis e a localização membranar o conjunto de estados ocultos.

Na situação explorada a seguir, pretendemos desenvolver um fármaco para o HIV, antagonista do co-recetor transmembranar CCR5, que serve de local de ligação para este vírus e permite a sua entrada nas células.
(Ser antagonista é ser um "bloqueador" que se liga a um recetor e impede que outras moléculas o ativem)

Sabemos que o HIV se liga à parte extracelular da proteína, por isso, vamos tentar prever a sua topologia para nos ajudar a desenvolver um fármaco eficiente


In [34]:
sequencia: SeqRecord = SeqIO.read("rcsb_pdb_4MBS.fasta", "fasta")
SeqIO.write([sequencia], sys.stdout, "fasta")
CCR5_seq = str(sequencia.seq)

>4MBS_1|Chains A, B|Chimera protein of C-C chemokine receptor type 5 and Rubredoxin|Homo sapiens (9606)
GAPDYQVSSPIYDINYYTSEPCQKINVKQIAARLLPPLYSLVFIFGFVGNMLVILILINY
KRLKSMTDIYLLNLAISDLFFLLTVPFWAHYAAAQWDFGNTMCQLLTGLYFIGFFSGIFF
IILLTIDRYLAVVHAVFALKARTVTFGVVTSVITWVVAVFASLPNIIFTRSQKEGLHYTC
SSHFPYSQYQFWKNFQTLKIVILGLVLPLLVMVICYSGILKTLLRMKKYTCTVCGYIYNP
EDGDPDNGVNPGTDFKDIPDDWVCPLCGVGKDQFEEVEEEKKRHRDVRLIFTIMIVYFLF
WAPYNIVLLLNTFQEFFGLNNCSSSNRLDQAMQVTETLGMTHCCINPIIYAFVGEEFRNY
LLVFFQKHIAKRFCKCCSIFQQEAPERASSVYTRSTGEQEISVGLGRPLEVLFQ


Uma primeira abordagem seria criar um modelo como este a seguir e treiná-lo com sequências de proteínas transmembranares conhecidas. <br>
O [PDBTM](https://pdbtm.unitmp.org/) é uma base de dados semelhante ao PDB mas focado em proteínas transmembranares, portanto podemos recolher as sequências daí

In [35]:
hmm_prot = HiddenMarkovModel(states="iMo",                                  # inicializar com uma matrix 
                             values=IUPACData.extended_protein_letters,     # de transmissão alearória;
                             initial_distribution=[0.5, 0, 0.5],            # sugerir que não começa na membrana
                             seed=3)

# sugerir que os estados 'i' e 'o' não estão conectados
hmm_prot.tmat[0,2] = 0
hmm_prot.tmat[2,0] = 0

Para recolher as sequências, utilizámos o ficheiro xml com todas as entradas

In [36]:
import xml.etree.ElementTree as ET
from xml.etree.ElementTree import ElementTree, Element
pdbtm_tree = ET.parse("pdbtm_all.xml")

In [37]:
def recolher_sequencias(tree: ElementTree) -> list[str]:
    sequencias = []
    percorrer_arvore(tree.getroot(), sequencias)
    return sequencias

def percorrer_arvore(elem: Element, seqlist: list):
    if elem is None:
        return
    
    for child in elem:
        if child.tag.endswith("SEQ"):
            # Há algumas sequências que começam com "-" e terminam com "?"
            # X_TODO descobrir porquê - inconclusivo, em uma sequência há um segmento
            # PH?SPHATIDYLSERINESYNTHASE, em outra, um que diz ST?P
            # talvez sejam dados mal inputados, mas há outras que têm simplesmente
            # uma forma -XXXX? ou ?XXXX
            # há outra em que aparece           VRR?EAI?HG?PFQ
            # cuja correspondência no pdbtm é   ---SEAITHGTPFQ
            # deste modo, será melhor retirar as sequências com estes caracteres
            seq = "".join(child.text.split())#.removeprefix("-").removesuffix("?")
                    
            if "?" in seq or "-" in seq:
                continue
                #pass

            seqlist.append(seq)
        percorrer_arvore(child, seqlist)

seqs = recolher_sequencias(pdbtm_tree)

In [38]:
# temos 63919 sequências com as quais treinar mas não temos todo o tempo do mundo
NSEQS_PARA_TREINO = 100
seqs_treino = np.random.choice(seqs, NSEQS_PARA_TREINO, replace=False)

Com as nossas sequências organizadas, podemos treinar o modelo com o algoritmo de Baum-Welch

In [39]:
print(hmm_prot.baum_welch_algorithm(seqs_treino, verbose=2))

loop 0
seq 0: gamas e csis calculados
todos os gamas e csis calculados
loop 0
seq 0: gamas e csis calculados
todos os gamas e csis calculados
HMM treinado com sucesso, parâmetros finais:

Matriz de transmissão
         i        M        o
i  0.44268  0.12028  0.43704
M  0.17139  0.10545  0.72316
o  0.86786  0.05102  0.08112

Matriz de emissão
         A        C        D        E        F        G        H        I  \
i  0.08329  0.02050  0.03518  0.06327  0.04247  0.05296  0.00127  0.00998   
M  0.32539  0.01358  0.09179  0.18164  0.19478  0.15975  0.00377  0.01010   
o  0.00686  0.00231  0.04687  0.00775  0.02580  0.09591  0.05720  0.16156   

         K        L  ...        T        V        W        Y    B    X    Z  \
i  0.00678  0.10003  ...  0.04700  0.10304  0.01128  0.06012  0.0  0.0  0.0   
M  0.00327  0.00045  ...  0.01235  0.00037  0.00014  0.00080  0.0  0.0  0.0   
o  0.12322  0.14151  ...  0.07447  0.03737  0.03192  0.00210  0.0  0.0  0.0   

     J        U    O  
i  0.0

In [40]:
hmm_prot.viterbi_algorithm(CCR5_seq)

Logaritmo da probablilidade: -582.3864193077173

  1 GAPDYQVSSPIYDINYYTSEPCQKINVKQIAARLLPPLYSLVFIFGFVGNMLVILILINY
  1 oiioioiiiioiMoiiioiiiiiooiioioiiioiiioiioiMoioiioioiioioioii

 61 KRLKSMTDIYLLNLAISDLFFLLTVPFWAHYAAAQWDFGNTMCQLLTGLYFIGFFSGIFF
 61 oiioioiMoioiioioiMoiMoioiiMoioiiiiioiMoiioiioioioiMoiMoiMoiM

121 IILLTIDRYLAVVHAVFALKARTVTFGVVTSVITWVVAVFASLPNIIFTRSQKEGLHYTC
121 ooioioiiioiiioiioiioiioioioiioiioioiiMoiiioiiooioiiioioioioi

181 SSHFPYSQYQFWKNFQTLKIVILGLVLPLLVMVICYSGILKTLLRMKKYTCTVCGYIYNP
181 iioiiiioioiioiMoiiooioiMoioioiioioiiiMoioioiioioioioiioioioi

241 EDGDPDNGVNPGTDFKDIPDDWVCPLCGVGKDQFEEVEEEKKRHRDVRLIFTIMIVYFLF
241 MoioioioioioioioioioioiiioioiMoioiiiiiiMooioioiiioiioioiiMoi

301 WAPYNIVLLLNTFQEFFGLNNCSSSNRLDQAMQVTETLGMTHCCINPIIYAFVGEEFRNY
301 oiiiioioioioioiMMoioiiiiioioioioiioioiMoioiioiiooiMoioiMoioi

361 LLVFFQKHIAKRFCKCCSIFQQEAPERASSVYTRSTGEQEISVGLGRPLEVLFQ
361 oiiMoioioioioioiiioioiiiiiiMoiiioiioiMoioiioioiioiioii




Como podemos ver, tanto pelo facto da matriz de transmissão ter valores não nulos para as transições entre `i` e `o`, como pelos resultados que dizem que os aminoácidos saltam através da membrana como se ela fosse do tamanho de uma ligação peptídica, este modelo não representa bem a realidade da relação entre aminoácidos e topologia membranar: é simples demais.

O que os modelos reais fazem de diferente é forçar as regiões da proteína a terem um comprimento mínimo através de estados com apenas uma transição. Desta forma, se o algoritmo entrar na membrana, tem que percorrer os estados de membrana todos até que possa chegar ao outro lado, prevenindo estes saltos de um aminoácido transmembranar

### Modelo Phobius (https://phobius.sbc.su.se/data.html)

O Modelo Phobius é um HMM feito para este mesmo propósito cujos parâmetros estão disponíveis para download. Conta com 189 estados ocultos incluindo um grupo de estados destinados especificamente a prever péptidos sinal


![image.png](phobius_cadeia.jpg)


Vamos utilizá-lo para ver o que nos diz sobre a nossa proteína

In [ ]:
a, b, states, vals, pi, labels = parse_phobius_model()
labels.update({"met1": "n"})    # no modelo, a label de met1 é "NULL"

hmm_phobius = HiddenMarkovModel(a, b, states=states, values=vals, initial_distribution=pi)

In [42]:
hmm_phobius.viterbi_algorithm(CCR5_seq, labels)

Logaritmo da probablilidade: -525.667894694714

  1 GAPDYQVSSPIYDINYYTSEPCQKINVKQIAARLLPPLYSLVFIFGFVGNMLVILILINY
  1 oooooooooooooooooooooooooooooooooooooMMMMMMMMMMMMMMMMMMMMMii

 61 KRLKSMTDIYLLNLAISDLFFLLTVPFWAHYAAAQWDFGNTMCQLLTGLYFIGFFSGIFF
 61 iiiiiiiiiMMMMMMMMMMMMMMMMMMMMoooooooooooooooooooMMMMMMMMMMMM

121 IILLTIDRYLAVVHAVFALKARTVTFGVVTSVITWVVAVFASLPNIIFTRSQKEGLHYTC
121 MMMMMMMMMMMiiiiiiiiiiiMMMMMMMMMMMMMMMMMMMMMMMMMMoooooooooooo

181 SSHFPYSQYQFWKNFQTLKIVILGLVLPLLVMVICYSGILKTLLRMKKYTCTVCGYIYNP
181 oooooooooooooooooooMMMMMMMMMMMMMMMMMMMMiiiiiiiiiiiiiiiiiiiii

241 EDGDPDNGVNPGTDFKDIPDDWVCPLCGVGKDQFEEVEEEKKRHRDVRLIFTIMIVYFLF
241 iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiMMMMMMMMMMMM

301 WAPYNIVLLLNTFQEFFGLNNCSSSNRLDQAMQVTETLGMTHCCINPIIYAFVGEEFRNY
301 MMMMMMMMMMOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO

361 LLVFFQKHIAKRFCKCCSIFQQEAPERASSVYTRSTGEQEISVGLGRPLEVLFQ
361 OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO




Agora, com as previsões da topologia da proteína podemos usá-las para melhor informar o desenvolvimento do nosso fármaco.

---
---

#### Simplificação do modelo para os Macro-estados *Inside, Membrane, Outside*

Se ainda assim quisermos testar até que ponto nosso primeiro modelo simples é utilizável, podemos agrupar os estados que se referem a um mesmo local e calcular os parâmetros de um Phobius simplificado:

In [ ]:
indexer = []
for state in states:
    label = labels[state]
    if label in ["i", "n"]:
        indexer.append("i")
    elif label in ["M", "h", "c"]:
        indexer.append("M")
    else:
        indexer.append("o")

N = hmm_phobius.nstates
sides = ["i", "M", "o"]
aas = hmm_phobius.values[:20]

simple_tmat: Matrix2D   = np.zeros((3,3))
simple_emat: Emission2D = np.zeros((3,20))
simple_init: Vector1D   = np.zeros((3))

# para calcular a nova distribuição inicial, simplesmente somamos as probabilidades
# dependendo na sua label. Porêm para os outros parâmetros (as matrizes de transição
# e emissão), é necessário fazer uma média ponderada dos valores que queremos tendo
# em conta o valor do estado estacionário para que um estado com mais prevalência
# ao longo do tempo não tenha o mesmo peso que um estado que acontece uma vez a cada 1000 anos
for i, side1 in enumerate(sides):

    indices_side1 = [idx for idx in range(N) if indexer[idx] == side1]

    simple_init[i] = np.sum(hmm_phobius.initial_dist[indices_side1])

    total_pi_side1 = np.sum(hmm_phobius.stationary_state[indices_side1])
    if total_pi_side1 == 0: continue

    for j, side2 in enumerate(sides):
        indices_side2 = [idx for idx in range(N) if indexer[idx] == side2]

        weighted_trans_sum = 0
        for st_i in indices_side1:
            weighted_trans_sum += \
                np.sum(hmm_phobius.tmat[st_i, indices_side2]) * \
                hmm_phobius.stationary_state[st_i] / total_pi_side1
        
        simple_tmat[i,j] = weighted_trans_sum
        
    for k, aa in enumerate(aas):
        
        weighted_emission_sum = 0
        for st_i in indices_side1:
            weighted_emission_sum += \
                np.sum(hmm_phobius.emat[st_i, k]) * \
                hmm_phobius.stationary_state[st_i] / total_pi_side1
        
        simple_emat[i,k] = weighted_emission_sum


indexed_simple_init = pd.Series(simple_init.round(3), index=sides)
indexed_simple_emat = pd.DataFrame(simple_emat.round(3), index=sides, columns=aas)
indexed_simple_tmat = pd.DataFrame(simple_tmat.round(3), index=sides, columns=sides)
simplified_phobius = HiddenMarkovModel(simple_tmat, simple_emat, states=sides, values=aas, initial_distribution=simple_init)

In [44]:
simplified_phobius.viterbi_algorithm(CCR5_seq)

Logaritmo da probablilidade: -532.4289086518869

  1 GAPDYQVSSPIYDINYYTSEPCQKINVKQIAARLLPPLYSLVFIFGFVGNMLVILILINY
  1 oooooooooooooooooooooooooooooooooooooMMMMMMMMMMMMMMMMMMMMMii

 61 KRLKSMTDIYLLNLAISDLFFLLTVPFWAHYAAAQWDFGNTMCQLLTGLYFIGFFSGIFF
 61 iiiiiiiiMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMM

121 IILLTIDRYLAVVHAVFALKARTVTFGVVTSVITWVVAVFASLPNIIFTRSQKEGLHYTC
121 MMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMiiiiiiiiiiii

181 SSHFPYSQYQFWKNFQTLKIVILGLVLPLLVMVICYSGILKTLLRMKKYTCTVCGYIYNP
181 iiiiiiiiiiiiiiiiiiiMMMMMMMMMMMMMMMMMMMMMiiiiiiiiiiiiiiiiiiii

241 EDGDPDNGVNPGTDFKDIPDDWVCPLCGVGKDQFEEVEEEKKRHRDVRLIFTIMIVYFLF
241 iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiMMMMMMMMMMMM

301 WAPYNIVLLLNTFQEFFGLNNCSSSNRLDQAMQVTETLGMTHCCINPIIYAFVGEEFRNY
301 MMMMMMMMMMiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii

361 LLVFFQKHIAKRFCKCCSIFQQEAPERASSVYTRSTGEQEISVGLGRPLEVLFQ
361 iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii




Comparando com o  original:

In [45]:
p, phobius_hidden_seq = hmm_phobius.viterbi_algorithm(CCR5_seq, labels, use_results=True)
p, simple_hidden_seq = simplified_phobius.viterbi_algorithm(CCR5_seq, use_results=True)
np.array(phobius_hidden_seq) == np.array(simple_hidden_seq)

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True, False,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

Este Phobius simplificado é o que usámos como base para os cálculos do hands-on! Depois de tentar resolver os exercícios, podes usar a célula a seguir para confirmar os resultados

In [46]:
ex_start = CCR5_seq.index("TRSQ")
p, ex_hid_seq, CCR5_seq_deltas, CCR5_seq_psis = simplified_phobius.viterbi_algorithm(CCR5_seq[:ex_start+4], use_results=True, use_matrices=True)

simplified_phobius.viterbi_algorithm(CCR5_seq[:ex_start+4])

print("seq a.a. -", CCR5_seq[ex_start-1:ex_start+4])
print("", *simplified_phobius.states, sep="           ")
print("delta")
print(CCR5_seq_deltas[ex_start-1:ex_start+4], sep="\n", end="\n\n")
print("psi")
print(CCR5_seq_psis[ex_start-1:ex_start+4], sep="\n", end="\n\n")

print("seq estados - ",*ex_hid_seq[ex_start-1:ex_start+4], sep="")

Logaritmo da probablilidade: -216.7154944850862

  1 GAPDYQVSSPIYDINYYTSEPCQKINVKQIAARLLPPLYSLVFIFGFVGNMLVILILINY
  1 oooooooooooooooooooooooooooooooooooooMMMMMMMMMMMMMMMMMMMMMii

 61 KRLKSMTDIYLLNLAISDLFFLLTVPFWAHYAAAQWDFGNTMCQLLTGLYFIGFFSGIFF
 61 iiiiiiiiMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMM

121 IILLTIDRYLAVVHAVFALKARTVTFGVVTSVITWVVAVFASLPNIIFTRSQ
121 MMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMiiii


seq a.a. - FTRSQ
           i           M           o
delta
[[-212.10527179 -210.17336349 -212.0900429 ]
 [-212.95532401 -211.52472821 -212.94225162]
 [-214.25985441 -213.77711999 -214.37220757]
 [-215.3838473  -215.15205172 -215.49059549]
 [-216.71549449 -217.29265468 -216.84681123]]

psi
[[0 1 2]
 [1 1 1]
 [0 1 2]
 [0 1 2]
 [0 1 2]]

seq estados - Miiii


O Phobius não é o único HMM desenvolvido para prever topologia de membrana, tal como qualquer outro algoritmo em Machine Learning, a escolha dos hiperparâmetros é muito importante para o sucesso dos modelos. No caso dos HMMs os hiperparâmetros consistem no espaço de estados $S$, o espaço de símbolos $V$, e, provavelmente o mais importante, a estrutura da Cadeia subjacente ao modelo e como ela interage com os símbolos.

Convidamos-te a acessar o [HMMTOP](https://hmmtop.pbrg.hu/index.php) para ver as diferenças entre resultados.

---
---
